# W4-T2 -- RVC voice conversion: attack CM02

Trains a small per-speaker RVC voice model for each of ~12 train-pool speakers,
then converts real speech from *other* train-pool speakers into each target's
voice -- 12 x 125 = ~1,500 clips, the CM02 target in `docs/attack_taxonomy.md`.

**Why this exists.** P-019 (`docs/problems_and_decisions.md`) measured that XTTS-v2
(CM01) compresses pitch range by ~35% relative to real speech -- it invents
prosody from text and regresses to a flat contour. That makes XTTS a legitimate
but *easy* attack. RVC starts from real speech, so the pitch contour is human and
that compression cannot happen -- it is the second, harder attack family the plan
was written around.

**Why Kaggle, not the dev laptop.** P-017/P-018 found the RVC toolchain installs
cleanly once you are *not* on Windows fighting MSVC Build Tools -- the real wall
was `fairseq` needing to compile, which Linux (this container) does not have the
same fight over. Training and conversion both belong here.

---

## Before you run

1. **Settings -> Accelerator -> GPU T4 x2** (or P100). **Internet -> On.**
2. **+ Add Input -> your MUCS 2021 train-pool dataset.** Upload
   `C:\dfdata\raw\mucs2021` preserving the `raw/mucs2021/...` folder structure --
   the project's `src.utils.paths.resolve()` re-roots `data/manifests/clip_index.csv`
   onto whatever `DATA_ROOT` you point at, by finding that `raw/` anchor, so no
   custom repackaging is needed. Only the **train pool** (25 speakers) is used, so
   a train-pool-only subset upload is enough if the full corpus does not fit.
3. **Add-ons -> Secrets -> `MENTOR_SIGNOFF`** is *not a thing Kaggle supports for
   files*. Instead: **+ Add Input -> a small private dataset containing the signed
   ethics PDF** (`docs/ethics/mentor_signoff_2026-08-12.pdf` is gitignored on
   purpose -- it carries real signatures and this repo is public -- so a fresh
   clone never has it, and the ethics gate blocks generation until you supply
   it here).
4. This repo is **public**, so no `GH_PAT` secret is needed to clone it.

Roughly 5-10 min/target to train on a T4 at the smoke-test epoch count below;
budget more for the full 100-epoch run across 12 targets.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH    = "week8-krishna-rvc-generation"

RVC_WEBUI_REPO = "https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git"

# Smoke test first: 1 target, few epochs, confirm the whole chain (preprocess ->
# f0 -> feature -> train -> convert) actually runs before committing a T4 session
# to the full 12-target run.
N_TARGETS = 1
TRAIN_EPOCHS = 20
CONVERSIONS_PER_TARGET = 10

# Full run values (set these once the smoke test's cell 9 output looks right):
# N_TARGETS = 12
# TRAIN_EPOCHS = 100
# CONVERSIONS_PER_TARGET = 125

## 2. Clone the repo

Public repo, no PAT needed (verified: `api.github.com/repos/...` returns 200
unauthenticated).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"


def sh(cmd, cwd=None, check=True):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd, text=True, capture_output=True)
    if r.stdout.strip():
        print(r.stdout[-3000:])
    if r.returncode != 0:
        if r.stderr.strip():
            print(r.stderr[-3000:])
        if check:
            raise SystemExit("command failed with exit %d" % r.returncode)
    return r


if REPO.exists():
    import shutil
    shutil.rmtree(REPO)
sh(["git", "clone", "--branch", BRANCH, "--depth", "1", "https://%s" % REPO_HOST, str(REPO)])
os.chdir(REPO)
sys.path.insert(0, str(REPO))
sh("git log --oneline -3")


## 3. Ethics sign-off

The gate has no override (`src/data/ethics_gate.py`) -- this is deliberate, not a
formality. Copy the signed PDF from your Kaggle input dataset in before anything
else runs.

In [ ]:
import glob
import shutil

signoff_candidates = glob.glob("/kaggle/input/**/mentor_signoff*.pdf", recursive=True)
if not signoff_candidates:
    raise SystemExit(
        "no mentor_signoff*.pdf found under /kaggle/input -- add the ethics-signoff "
        "dataset as an Input (see 'Before you run', step 3) and re-run"
    )
Path("docs/ethics").mkdir(parents=True, exist_ok=True)
dest = Path("docs/ethics") / Path(signoff_candidates[0]).name
shutil.copy2(signoff_candidates[0], dest)
print("copied ->", dest)

from src.data.ethics_gate import signoff_status
status = signoff_status()
print(status.describe())
if not status.signed:
    raise SystemExit("ethics gate still closed -- check the copied file is not a 0-byte placeholder")


## 4. Locate the MUCS train-pool audio + resolve manifest paths

`data/manifests/clip_index.csv` was written on the dev laptop with
`C:\dfdata\raw\mucs2021\...` paths. `src.utils.paths.resolve()` rebases each path
onto the local `DATA_ROOT` by finding the `raw/` anchor -- exactly the portability
mechanism this project already uses for Colab/Kaggle (see the module docstring).

In [ ]:
import pandas as pd

from src.utils.paths import resolve_column

mucs_roots = glob.glob("/kaggle/input/**/raw/mucs2021", recursive=True)
if not mucs_roots:
    print("contents of /kaggle/input:")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print("  ", p)
    raise SystemExit("raw/mucs2021 not found under any /kaggle/input dataset -- see step 2 above")

# DATA_ROOT is the directory that CONTAINS raw/, i.e. one level above raw/mucs2021.
DATA_ROOT = str(Path(mucs_roots[0]).parent.parent)
os.environ["DATA_ROOT"] = DATA_ROOT
print("DATA_ROOT =", DATA_ROOT)

index = pd.read_csv("data/manifests/clip_index.csv")
pools = pd.read_csv("data/manifests/speaker_pools.csv")
index = resolve_column(index, "wav_path", root=DATA_ROOT)

train_pool = set(pools.loc[pools["pool"] == "train", "speaker"].astype(str))
print("train-pool speakers:", len(train_pool))

# Spot-check a sample of train-pool clips actually resolve to real files.
sample = index[index["speaker"].astype(str).isin(train_pool)].sample(
    min(20, len(index)), random_state=0
)
missing = [p for p in sample["wav_path"] if not Path(p).is_file()]
print("spot-checked %d train-pool clips, %d missing" % (len(sample), len(missing)))
if missing:
    print("first missing path:", missing[0])
    raise SystemExit("resolved paths do not exist -- check the uploaded dataset's folder structure")


## 5. Install the RVC toolchain

Training is the RVC-WebUI repo's own scripts, not a pip package (P-017/P-018), and
conversion is that same checkout's `infer/cli.py` -- one tree, one set of weights,
nothing to version-match by hand.

**What changed since P-018 was written.** Upstream dropped `fairseq` entirely:
HuBERT is a Transformers model directory now (`infer/hubert.py`), and the training
scripts moved from `infer/modules/train/` to `train/`. So the `omegaconf`/`pip<24.1`
dance and the whole MSVC saga in P-018 no longer apply -- there is nothing left to
compile.

`requirments_*.txt` is **not** installed here on purpose. Those files pin
`numpy<2`, `pydantic<2`, `gradio 3.14` and a China PyPI mirror to keep `webui.py`'s
Gradio 3 UI working; we never launch the UI, and applying those pins to a Kaggle
image would tear out the preinstalled torch stack. The four packages below are what
the training and inference scripts actually import and Kaggle does not already ship.

In [ ]:
RVC_REPO = str(WORK / "rvc-webui")
if not Path(RVC_REPO).exists():
    sh(["git", "clone", "--depth", "1", RVC_WEBUI_REPO, RVC_REPO])
os.environ["RVC_REPO"] = RVC_REPO

# Only what the five scripts we run import and Kaggle lacks. Everything else they
# need (torch, numpy, scipy, librosa, soundfile, transformers, tensorboard,
# scikit-learn, matplotlib) is already in the Kaggle image.
sh([sys.executable, "-m", "pip", "install", "-q",
    "av", "ffmpeg-python", "praat-parselmouth", "faiss-cpu"])

import torch
print("torch     ", torch.__version__)
print("cuda avail", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("no GPU -- set Accelerator to GPU T4 x2 and restart the session")

# Fail here rather than three stages into the first target: these are exactly the
# imports preprocess / f0 / feature / index extraction do.
import faiss, parselmouth, soundfile, transformers  # noqa: F401
import av  # noqa: F401
print("faiss / parselmouth / av / transformers import OK")

## 6. Pretrained base models

RVC-WebUI ships none of its weights in git. Four things are needed and all four are
non-optional:

- **`assets/hubert_base/`** -- the feature extractor. A *directory* (config.json,
  preprocessor_config.json, pytorch_model.bin), not the old fairseq `hubert_base.pt`.
  Downloading a single `.pt` to the old path is the failure that looks like it
  worked and then raises `FileNotFoundError` during feature extraction.
- **`assets/rmvpe/rmvpe.pt`** -- the f0 estimator.
- **`assets/pretrained_v2/f0{G,D}40k.pth`** -- the generator/discriminator pair
  training warm-starts from. Each target here has minutes, not hours, of audio --
  far below a from-scratch budget -- so the warm start is what makes a per-speaker
  model viable at all.
- **`logs/mute/`** -- the silence example every training run's `filelist.txt`
  references. Missing it fails inside the dataloader, after preprocessing.

`download_rvc_assets` fetches exactly these (~700 MB, skipping whatever is already
there) and then re-runs the preflight, so a broken download surfaces now rather
than forty minutes into a session.

In [ ]:
from src.data import rvc_generation as rvc

fetched = rvc.download_rvc_assets(RVC_REPO)
for path in fetched:
    print("fetched", path)
rvc.assert_rvc_assets(RVC_REPO)
print("\nRVC assets present and consistent")

## 7. Select RVC targets + build the conversion job table

The firewall is checked here, before any GPU time: both endpoints of every
conversion -- target *and* source -- must be train-pool speakers, no speaker is
converted into themselves, and no source clip is reused within one target. A
converted clip carries the **target's** speaker id, because the voice it holds is
the target's; that is what makes CM02's pool disjointness mean the same thing as
CM01's.

In [ ]:
config = rvc.RVCConfig(
    n_targets=N_TARGETS,
    conversions_per_target=CONVERSIONS_PER_TARGET,
    train_epochs=TRAIN_EPOCHS,
)

targets = rvc.select_target_speakers(index, pools, config)
print("%d target speaker(s) qualify (>= %.0fs each):" % (len(targets), config.min_training_seconds))
print(targets.to_string(index=False))
if targets.empty:
    raise SystemExit("no train-pool speaker has enough audio -- lower min_training_seconds or check DATA_ROOT")

jobs = rvc.build_rvc_jobs(
    index, pools, out_dir=str(WORK / "rvc_outputs"), config=config, targets=targets
)
Path("data/manifests").mkdir(parents=True, exist_ok=True)
jobs.to_csv("data/manifests/rvc_generation_jobs.csv", index=False)
print(rvc.rvc_job_summary(jobs))

## 8. Train each target's RVC voice model

Runs the 4-stage RVC-WebUI pipeline per target: preprocess -> f0 extract ->
feature extract -> train. `TRAIN_EPOCHS = 20` above is a smoke value -- raise it
(and `N_TARGETS`) once this cell completes cleanly for one speaker.

In [ ]:
models = {}
for target_row in targets.itertuples(index=False):
    target = str(target_row.speaker)
    clips = rvc.gather_training_clips(index, target)
    print("training %s (%d clips, %.0fs)..." % (target, len(clips), clips["duration_seconds"].sum()))
    model = rvc.train_speaker_model(
        target, clips, str(WORK / "rvc_train"), RVC_REPO, config=config
    )
    print("  ", model.describe())
    models[target] = model


## 9. Convert: real speech from other train-pool speakers -> each target's voice

In [ ]:
inferencer = rvc.load_rvc_inferencer()


def progress(i, total, row):
    print("  [%d/%d] %s -> %s" % (i, total, row["source_speaker"], row["target_speaker"]))


metadata_path = str(WORK / "rvc_outputs" / "generation_metadata.jsonl")
failures = []
written = rvc.convert_batch(
    jobs, inferencer, models, metadata_path=metadata_path, on_progress=progress, failures=failures
)
print("\nconverted %d clip(s)" % len(written))
if failures:
    print("%d failure(s) -- see the printed [FAILED] lines above" % len(failures))


## 10. Collect the artefacts

In [ ]:
import json

kept = rvc.rewrite_metadata(metadata_path)
records = rvc.read_metadata(metadata_path)
print("metadata reconciled: %d record(s) matching files on disk" % kept)
stats = rvc.rvc_generation_stats(records)
print(json.dumps(stats, indent=2))

summary = {
    "task": "W4-T2 RVC generation (CM02)",
    "n_targets": len(models),
    "models": {t: m.describe() for t, m in models.items()},
    "stats": stats,
}
(WORK / "rvc_generation_summary.json").write_text(json.dumps(summary, indent=2))
print("\nDownload from this version's Output tab:")
print("  -", WORK / "rvc_generation_summary.json")
print("  -", metadata_path)
print("  -", "data/manifests/rvc_generation_jobs.csv (copy out manually if needed)")
print("  -", WORK / "rvc_outputs", "(the converted clips themselves)")
